In [1]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import numpy as np
import pandas as pd

import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# 0) 配置
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

# 直接读取你已生成的特征文件（xlsx）
FEATURE_FILE_GLOB = "./Malodors_Rule&FG_features.xlsx"
# 如果你的文件名不同，改成你的，例如：
# FEATURE_FILE_GLOB = "./Malodors_transformed_MORGAN_features.xlsx"

# 固定参数：Optuna 只搜索部分超参
BASE_XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
    multi_strategy="multi_output_tree",
)


# =========================
# 1) XGBoost 版本检查
# =========================
def _ver_tuple(v: str):
    parts = re.split(r"[.+-]", v.strip())
    nums = []

    for p in parts[:3]:
        try:
            nums.append(int(p))
        except Exception:
            nums.append(0)

    while len(nums) < 3:
        nums.append(0)

    return tuple(nums)


def check_xgb_version():
    v = _ver_tuple(xgb.__version__)

    if v < (1, 6, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 太旧：multi-label 需要 >=1.6"
        )

    if "multi_strategy" in BASE_XGB_PARAMS and v < (2, 0, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 不支持 multi_output_tree，需要 >=2.0。\n"
            f"请升级 xgboost，或删除 BASE_XGB_PARAMS 中的 multi_strategy。"
        )


# =========================
# 2) 读取特征文件 & 按固定列位置划分 SMILES / y / X
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))

    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )

    return cands[-1]


def load_Xy_from_feature_file(xlsx_path: str):
    """
    固定数据格式：
    第 1 列：SMILES
    第 2-25 列：24 个 y 标签
    第 26 列及之后：X 特征列

    说明：
    - 不再通过 0/1 自动判断 y 标签列；
    - 避免 Morgan bit、Rule、FG、StructKG 等 0/1 或数值特征被误识别为标签；
    - 适用于当前 24 标签任务。
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 26:
        raise ValueError(
            f"当前文件列数为 {df.shape[1]}，不足以按“1列SMILES + 24列标签 + 特征列”划分。"
        )

    # 固定列位置
    smiles_col = df.columns[0]
    label_cols = list(df.columns[1:25])
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(
            f"标签列数量应为 24，但当前识别为 {len(label_cols)}。"
        )

    if len(feature_cols) == 0:
        raise ValueError(
            "未识别到特征列。请确认第 26 列之后为 X 特征。"
        )

    # y 标签转为 0/1 整数
    y_df = df[label_cols].copy()
    y_df = y_df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    # 检查 y 标签是否为 0/1
    invalid_label_cols = []
    for c in label_cols:
        vals = set(pd.unique(y_df[c].dropna()))
        if not vals.issubset({0, 1}):
            invalid_label_cols.append(c)

    if invalid_label_cols:
        raise ValueError(
            "以下标签列不是标准 0/1 标签，请检查数据：\n"
            + "\n".join(map(str, invalid_label_cols))
        )

    # X 特征转为数值
    X_df = df[feature_cols].copy()
    X_df = X_df.apply(pd.to_numeric, errors="coerce").fillna(0)

    X = X_df.astype(np.float32).values
    y = y_df.values.astype(int)

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    print("[INFO] Column split mode: fixed-position")
    print(f"[INFO] SMILES column: {smiles_col}")
    print(f"[INFO] Label columns: columns 2-25, n={len(label_cols)}")
    print(f"[INFO] Feature columns: columns 26-end, n={len(feature_cols)}")

    print("\n[INFO] Label columns:")
    for i, c in enumerate(label_cols, start=1):
        print(f"  y{i:02d}: {c}")

    return X, y, feature_cols, label_cols, df


# =========================
# 3) 指标：macro
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }


def extract_positive_proba(p, n_labels: int):
    """
    兼容 XGBoost 多标签输出：

    常见情况：
    - multi_output_tree 下 predict_proba 通常返回形状为 (n_samples, n_labels)
    - 某些版本或设置下可能返回 list 或 3D array
    """
    if isinstance(p, list):
        out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)

        for k in range(n_labels):
            pk = p[k]

            if pk.ndim != 2:
                raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

            if pk.shape[1] == 2:
                out[:, k] = pk[:, 1].astype(np.float32)
            elif pk.shape[1] == 1:
                out[:, k] = pk[:, 0].astype(np.float32)
            else:
                raise ValueError(
                    f"predict_proba[{k}] 类别数异常: {pk.shape[1]}"
                )

        return out

    p = np.asarray(p)

    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)

    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)

        if p.shape[1] == n_labels:
            return p.astype(np.float32)

        raise ValueError(
            f"predict_proba 输出为二维，但列数 {p.shape[1]} 与标签数 {n_labels} 不一致。"
        )

    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")


# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    """
    多标签任务中没有直接的 StratifiedKFold。
    这里使用每个样本的标签数量 label cardinality 做近似分层。
    """
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)

    if len(uniq) <= 15:
        return card

    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(
        r,
        q=min(n_bins, len(np.unique(r))),
        labels=False,
        duplicates="drop"
    )

    return np.asarray(bins, dtype=int)


def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    return list(skf.split(X, strat_y))


# =========================
# 5) 5 折 CV 评估
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

        print(
            f"    [FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | "
            f"AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | "
            f"P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | "
            f"Spec={m['Specificity_macro']:.6f}"
        )

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}

    for k in keys:
        vals = [fm[k] for fm in fold_metrics]

        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]

    return mean_metrics


# =========================
# 6) Optuna 超参空间
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_XGB_PARAMS)

    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    })

    return params


# =========================
# 7) 主流程：读特征 -> 固定 folds -> Optuna -> 输出保存
# =========================
def main():
    check_xgb_version()

    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    X, y, feat_cols, label_cols, df_all = load_Xy_from_feature_file(feature_file)

    print(
        f"\n[INFO] X shape={X.shape} "
        f"(features={len(feat_cols)}) | "
        f"y shape={y.shape} "
        f"(labels={len(label_cols)})"
    )

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)

        print("\n" + "=" * 80)
        print(f"[TRIAL {trial.number:02d}] params:")
        print(json.dumps(params, indent=2, ensure_ascii=False, default=str))

        mean_metrics = cv_eval_one_paramset(
            X,
            y,
            folds,
            params,
            thresh=THRESH
        )

        score = mean_metrics["AUPRC_macro"]

        print(
            f"\n[TRIAL {trial.number:02d}] "
            f"score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []

    for t in study.trials:
        if t.value is None:
            continue

        row = {
            "trial": t.number,
            "AUPRC_macro": t.value,
            **t.params
        }

        row.update({
            k: v for k, v in t.user_attrs.items()
            if k.endswith("_macro")
        })

        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values(
        "AUPRC_macro",
        ascending=False
    )

    out_csv = "./optuna_singlemodel_XGB_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

    # 额外保存标签列和特征列，便于后续核查
    label_out = "./used_label_columns_24.csv"
    feature_out = "./used_feature_columns.csv"

    pd.DataFrame({"label_col": label_cols}).to_csv(
        label_out,
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame({"feature_col": feat_cols}).to_csv(
        feature_out,
        index=False,
        encoding="utf-8-sig"
    )

    print("[SAVED]", label_out)
    print("[SAVED]", feature_out)


if __name__ == "__main__":
    main()

[INFO] Using feature file: ./Malodors_Rule&FG_features.xlsx


/root/miniconda3/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
[I 2026-05-13 14:15:50,931] A new study created in memory with name: no-name-e79035e3-47e7-4584-9da9-b3334a0094d1


[INFO] Column split mode: fixed-position
[INFO] SMILES column: Canonical_SMILES
[INFO] Label columns: columns 2-25, n=24
[INFO] Feature columns: columns 26-end, n=379

[INFO] Label columns:
  y01: alcoholic
  y02: aldehydic
  y03: almond
  y04: aromatic
  y05: burnt
  y06: cabbage
  y07: cheesy
  y08: cherry
  y09: chocolate
  y10: ethereal
  y11: fishy
  y12: fruity
  y13: garlic
  y14: gassy
  y15: green
  y16: ketonic
  y17: musty
  y18: pungent
  y19: sharp
  y20: solvent
  y21: sour
  y22: sulfurous
  y23: sweaty
  y24: sweet

[INFO] X shape=(3756, 379) (features=379) | y shape=(3756, 24) (labels=24)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] params:
{
  "objective": "binary:logistic",
  "eval_metric": "logloss",
  "tree_method": "hist",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": 0,
  "multi_strategy": "multi_output_tree",
  "n_estimators": 637,
  "max_depth": 10,
  "learning_rate": 0.08960785365368121,
  "subsample": 0.8394633936788146,
  "colsample_bytree": 0.6624074561769746,
  "min_child_weight": 0.7978542347074177,
  "reg_lambda": 0.0017775399007348214,
  "reg_alpha": 0.3426417745118369,
  "gamma": 3.005575058716044
}
    [FOLD 1] AUPRC=0.407212 | AUROC=0.860238 | Acc=0.942265 | P=0.571261 | R=0.267732 | Spec=0.971174
    [FOLD 2] AUPRC=0.382839 | AUROC=0.860234 | Acc=0.940246 | P=0.540321 | R=0.257156 | Spec=0.971910
    [FOLD 3] AUPRC=0.381874 | AUROC=0.848936 | Acc=0.942688 | P=0.515970 | R=0.286313 | Spec=0.971252
    [FOLD 4] AUPRC=0.416589 | AUROC=0.853692 | Acc=0.939026 | P=0.563429 | R=0.258568 | Spec=0.968909
    [FOLD 5] AUPRC=0.415884 | AUROC=0.861479 | Acc=0.944407 | P=0.558871 | R=0.2

In [2]:
# -*- coding: utf-8 -*-
import os
import glob
import json
import warnings
import numpy as np
import pandas as pd

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multioutput import MultiOutputClassifier

# =========================
# 0) 全局：尽量屏蔽警告 + LightGBM 日志
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import lightgbm as lgb
except Exception as e:
    raise RuntimeError(
        "未检测到 lightgbm。请先安装：pip install lightgbm\n"
        f"原始错误：{repr(e)}"
    )

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

# 直接读取你已生成的特征文件
FEATURE_FILE_GLOB = "./Malodors_Rule&FG_features.xlsx"

# 固定参数：Optuna 只搜索部分超参
BASE_LGB_PARAMS = dict(
    objective="binary",
    boosting_type="gbdt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=-1,
)


# =========================
# 2) 读取特征文件 & 按固定列位置划分 SMILES / y / X
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))

    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )

    return cands[-1]


def load_Xy_from_feature_file(xlsx_path: str):
    """
    固定数据格式：
    第 1 列：SMILES
    第 2-25 列：24 个 y 标签
    第 26 列及之后：X 特征列

    说明：
    - 不再通过 0/1 自动判断 y 标签列；
    - 避免 Rule、FG、Morgan、StructKG 等 0/1 或数值特征被误识别为标签；
    - 适用于当前 24 标签任务。
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 26:
        raise ValueError(
            f"当前文件列数为 {df.shape[1]}，不足以按“1列SMILES + 24列标签 + 特征列”划分。"
        )

    # 固定列位置
    smiles_col = df.columns[0]
    label_cols = list(df.columns[1:25])
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(
            f"标签列数量应为 24，但当前识别为 {len(label_cols)}。"
        )

    if len(feature_cols) == 0:
        raise ValueError(
            "未识别到特征列。请确认第 26 列之后为 X 特征。"
        )

    # y 标签转为 0/1 整数
    y_df = df[label_cols].copy()
    y_df = y_df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    # 检查 y 标签是否为 0/1
    invalid_label_cols = []
    for c in label_cols:
        vals = set(pd.unique(y_df[c].dropna()))
        if not vals.issubset({0, 1}):
            invalid_label_cols.append(c)

    if invalid_label_cols:
        raise ValueError(
            "以下标签列不是标准 0/1 标签，请检查数据：\n"
            + "\n".join(map(str, invalid_label_cols))
        )

    # X 特征转为数值
    X_df = df[feature_cols].copy()
    X_df = X_df.apply(pd.to_numeric, errors="coerce").fillna(0)

    X = X_df.astype(np.float32).values
    y = y_df.values.astype(int)

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    print("[INFO] Column split mode: fixed-position")
    print(f"[INFO] SMILES column: {smiles_col}")
    print(f"[INFO] Label columns: columns 2-25, n={len(label_cols)}")
    print(f"[INFO] Feature columns: columns 26-end, n={len(feature_cols)}")

    print("\n[INFO] Label columns:")
    for i, c in enumerate(label_cols, start=1):
        print(f"  y{i:02d}: {c}")

    return X, y, feature_cols, label_cols, df


# =========================
# 3) 指标：macro
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }


def extract_positive_proba(p, n_labels: int):
    """
    MultiOutputClassifier.predict_proba 通常返回 list：
    list 长度 = 标签数，每个元素形状为 (n_samples, n_classes)。

    注意：
    - 若某个 fold 中某个标签训练集只有单类，LightGBM 可能只返回 1 列；
    - 这里做了兼容，避免 pi[:, 1] 越界。
    """
    if isinstance(p, list):
        out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)

        for k in range(n_labels):
            pk = p[k]

            if pk.ndim != 2:
                raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

            if pk.shape[1] == 2:
                out[:, k] = pk[:, 1].astype(np.float32)
            elif pk.shape[1] == 1:
                # 无法从 predict_proba 本身判断该单类是 0 还是 1；
                # 这里兜底使用该列概率。通常极少发生。
                out[:, k] = pk[:, 0].astype(np.float32)
            else:
                raise ValueError(
                    f"predict_proba[{k}] 类别数异常: {pk.shape[1]}"
                )

        return out

    p = np.asarray(p)

    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)

    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)

        if p.shape[1] == n_labels:
            return p.astype(np.float32)

        raise ValueError(
            f"predict_proba 输出为二维，但列数 {p.shape[1]} 与标签数 {n_labels} 不一致。"
        )

    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")


# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    """
    多标签任务中没有直接的 StratifiedKFold。
    这里使用每个样本的标签数量 label cardinality 做近似分层。
    """
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)

    if len(uniq) <= 15:
        return card

    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(
        r,
        q=min(n_bins, len(np.unique(r))),
        labels=False,
        duplicates="drop"
    )

    return np.asarray(bins, dtype=int)


def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    return list(skf.split(X, strat_y))


# =========================
# 5) 5 折 CV 评估：LightGBM + MultiOutput
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        base_est = lgb.LGBMClassifier(**params)

        # 每个标签一个 LGBM；
        # n_jobs=1 避免“外层并行 + 内层线程”导致线程过载
        clf = MultiOutputClassifier(base_est, n_jobs=1)

        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

        print(
            f"    [FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | "
            f"AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | "
            f"P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | "
            f"Spec={m['Specificity_macro']:.6f}"
        )

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}

    for k in keys:
        vals = [fm[k] for fm in fold_metrics]

        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]

    return mean_metrics


# =========================
# 6) Optuna 超参空间：LightGBM
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_LGB_PARAMS)

    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),

        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 20),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),

        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 50.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),

        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 5.0),
    })

    return params


# =========================
# 7) 主流程：读特征 -> 固定 folds -> Optuna -> 输出保存
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    X, y, feat_cols, label_cols, df_all = load_Xy_from_feature_file(feature_file)

    print(
        f"\n[INFO] X shape={X.shape} "
        f"(features={len(feat_cols)}) | "
        f"y shape={y.shape} "
        f"(labels={len(label_cols)})"
    )

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)

        print("\n" + "=" * 80)
        print(f"[TRIAL {trial.number:02d}] params:")
        print(json.dumps(params, indent=2, ensure_ascii=False, default=str))

        mean_metrics = cv_eval_one_paramset(
            X,
            y,
            folds,
            params,
            thresh=THRESH
        )

        score = mean_metrics["AUPRC_macro"]

        print(
            f"\n[TRIAL {trial.number:02d}] "
            f"score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []

    for t in study.trials:
        if t.value is None:
            continue

        row = {
            "trial": t.number,
            "AUPRC_macro": t.value,
            **t.params
        }

        row.update({
            k: v for k, v in t.user_attrs.items()
            if k.endswith("_macro")
        })

        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values(
        "AUPRC_macro",
        ascending=False
    )

    out_csv = "./optuna_singlemodel_LIGHTGBM_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

    # 额外保存标签列和特征列，便于核查
    label_out = "./used_label_columns_24.csv"
    feature_out = "./used_feature_columns.csv"

    pd.DataFrame({"label_col": label_cols}).to_csv(
        label_out,
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame({"feature_col": feat_cols}).to_csv(
        feature_out,
        index=False,
        encoding="utf-8-sig"
    )

    print("[SAVED]", label_out)
    print("[SAVED]", feature_out)


if __name__ == "__main__":
    main()

[INFO] Using feature file: ./Malodors_Rule&FG_features.xlsx


[I 2026-05-13 14:44:57,839] A new study created in memory with name: no-name-4e22d592-76e7-4253-a0f0-be6b7ed5ee33


[INFO] Column split mode: fixed-position
[INFO] SMILES column: Canonical_SMILES
[INFO] Label columns: columns 2-25, n=24
[INFO] Feature columns: columns 26-end, n=379

[INFO] Label columns:
  y01: alcoholic
  y02: aldehydic
  y03: almond
  y04: aromatic
  y05: burnt
  y06: cabbage
  y07: cheesy
  y08: cherry
  y09: chocolate
  y10: ethereal
  y11: fishy
  y12: fruity
  y13: garlic
  y14: gassy
  y15: green
  y16: ketonic
  y17: musty
  y18: pungent
  y19: sharp
  y20: solvent
  y21: sour
  y22: sulfurous
  y23: sweaty
  y24: sweet

[INFO] X shape=(3756, 379) (features=379) | y shape=(3756, 24) (labels=24)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] params:
{
  "objective": "binary",
  "boosting_type": "gbdt",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": -1,
  "n_estimators": 937,
  "learning_rate": 0.17254716573280354,
  "num_leaves": 195,
  "max_depth": 12,
  "min_child_samples": 16,
  "subsample": 0.662397808134481,
  "colsample_bytree": 0.6232334448672797,
  "reg_lambda": 11.752647960576219,
  "reg_alpha": 0.002570603566117598,
  "min_split_gain": 3.540362888980227
}
    [FOLD 1] AUPRC=0.286916 | AUROC=0.820202 | Acc=0.938165 | P=0.836646 | R=0.107284 | Spec=0.980877
    [FOLD 2] AUPRC=0.300888 | AUROC=0.842478 | Acc=0.936862 | P=0.792235 | R=0.107525 | Spec=0.981727
    [FOLD 3] AUPRC=0.292054 | AUROC=0.807041 | Acc=0.939359 | P=0.757033 | R=0.108597 | Spec=0.982798
    [FOLD 4] AUPRC=0.289374 | AUROC=0.807137 | Acc=0.935697 | P=0.775684 | R=0.111577 | Spec=0.980051
    [FOLD 5] AUPRC=0.291433 | AUROC=0.816936 | Acc=0.938637 | P=0.763864 | R=0.109170 | Spec=0.979209

[TRIAL 00] score(AUPRC_macro)=0.292133 

In [3]:
# -*- coding: utf-8 -*-
import os
import glob
import json
import warnings
import numpy as np
import pandas as pd

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

# =========================
# 0) 配置 & 尽量屏蔽警告
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# 读取特征文件
FEATURE_FILE_GLOB = "./Malodors_Rule&FG_features.xlsx"

# RandomForest 固定参数，Optuna 只搜索部分超参
BASE_RF_PARAMS = dict(
    n_jobs=-1,
    random_state=RANDOM_SEED,
)


# =========================
# 1) 读取特征文件 & 按固定列位置划分 SMILES / y / X
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]


def load_Xy_from_feature_file(xlsx_path: str):
    """
    固定数据格式：
    第 1 列：SMILES
    第 2-25 列：24 个 y 标签
    第 26 列及之后：X 特征列

    说明：
    - 不再通过 0/1 自动判断 y 标签列；
    - 避免 Morgan bit、Rule、FG 等 0/1 特征被误识别为标签；
    - 适用于当前 24 标签任务。
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 26:
        raise ValueError(
            f"当前文件列数为 {df.shape[1]}，不足以按“1列SMILES + 24列标签 + 特征列”划分。"
        )

    # 固定列位置
    smiles_col = df.columns[0]
    label_cols = list(df.columns[1:25])
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(f"标签列数量应为 24，但当前识别为 {len(label_cols)}。")

    if len(feature_cols) == 0:
        raise ValueError("未识别到特征列。请确认第 26 列之后为 X 特征。")

    # y 标签转为 0/1 整数
    y_df = df[label_cols].copy()
    y_df = y_df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

    # 检查 y 标签是否为 0/1
    invalid_label_cols = []
    for c in label_cols:
        vals = set(pd.unique(y_df[c].dropna()))
        if not vals.issubset({0, 1}):
            invalid_label_cols.append(c)

    if invalid_label_cols:
        raise ValueError(
            "以下标签列不是标准 0/1 标签，请检查数据：\n"
            + "\n".join(map(str, invalid_label_cols))
        )

    # X 特征转为数值
    X_df = df[feature_cols].copy()
    X_df = X_df.apply(pd.to_numeric, errors="coerce").fillna(0)

    X = X_df.astype(np.float32).values
    y = y_df.values.astype(int)

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    print("[INFO] Column split mode: fixed-position")
    print(f"[INFO] SMILES column: {smiles_col}")
    print(f"[INFO] Label columns: columns 2-25, n={len(label_cols)}")
    print(f"[INFO] Feature columns: columns 26-end, n={len(feature_cols)}")

    print("\n[INFO] Label columns:")
    for i, c in enumerate(label_cols, start=1):
        print(f"  y{i:02d}: {c}")

    return X, y, feature_cols, label_cols, df


# =========================
# 2) 指标：macro
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }


def extract_positive_proba(p, n_labels: int, classes_list=None):
    """
    兼容：
    - multi-output RF 的 predict_proba: list，长度=labels，每个元素形状为 (n, n_classes_k)
    - 某标签在该 fold 训练集里只有单类时，predict_proba 可能为 (n, 1)
    """
    if isinstance(p, list):
        out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)

        for k in range(n_labels):
            pk = p[k]

            if pk.ndim != 2:
                raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

            if pk.shape[1] == 2:
                # 优先按 classes_ 找 class == 1 对应的概率列
                if classes_list is not None and len(classes_list) == n_labels:
                    cls = list(classes_list[k])
                    if 1 in cls:
                        j = cls.index(1)
                        out[:, k] = pk[:, j].astype(np.float32)
                    else:
                        out[:, k] = 0.0
                else:
                    out[:, k] = pk[:, 1].astype(np.float32)

            elif pk.shape[1] == 1:
                # 只有单一类别：若该类别是 1，则概率恒为 1；若是 0，则概率恒为 0
                if classes_list is not None and len(classes_list) == n_labels:
                    only_cls = int(list(classes_list[k])[0])
                    out[:, k] = 1.0 if only_cls == 1 else 0.0
                else:
                    out[:, k] = 0.0

            else:
                raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")

        return out

    # 非 list 情况，一般不会出现在 multi-output RF
    p = np.asarray(p)

    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)

    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)

    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")


# =========================
# 3) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    """
    多标签任务中没有直接的 StratifiedKFold。
    这里使用每个样本的标签数量 label cardinality 做近似分层。
    """
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)

    if len(uniq) <= 15:
        return card

    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(
        r,
        q=min(n_bins, len(np.unique(r))),
        labels=False,
        duplicates="drop"
    )

    return np.asarray(bins, dtype=int)


def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )
    return list(skf.split(X, strat_y))


# =========================
# 4) 5 折 CV 评估：RF
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = RandomForestClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(
            p,
            n_labels=n_labels,
            classes_list=getattr(clf, "classes_", None)
        )

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

        print(
            f"    [FOLD {fold_id}] "
            f"AUPRC={m['AUPRC_macro']:.6f} | "
            f"AUROC={m['AUROC_macro']:.6f} | "
            f"Acc={m['Accuracy_macro']:.6f} | "
            f"P={m['Precision_macro']:.6f} | "
            f"R={m['Recall_macro']:.6f} | "
            f"Spec={m['Specificity_macro']:.6f}"
        )

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}

    for k in keys:
        vals = [fm[k] for fm in fold_metrics]
        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]

    return mean_metrics


# =========================
# 5) Optuna 超参空间：RF
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_RF_PARAMS)

    max_depth_raw = trial.suggest_int("max_depth", 0, 40)
    max_depth = None if max_depth_raw == 0 else max_depth_raw

    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": max_depth,
        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", 0.3, 0.5, 0.8, 1.0]
        ),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "criterion": trial.suggest_categorical(
            "criterion",
            ["gini", "entropy", "log_loss"]
        ),
    })

    return params


# =========================
# 6) 主流程：读特征 -> 固定 folds -> Optuna -> 输出保存
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    X, y, feat_cols, label_cols, df_all = load_Xy_from_feature_file(feature_file)

    print(
        f"\n[INFO] X shape={X.shape} "
        f"(features={len(feat_cols)}) | "
        f"y shape={y.shape} "
        f"(labels={len(label_cols)})"
    )

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)

        print("\n" + "=" * 80)
        print(f"[TRIAL {trial.number:02d}] params:")
        print(json.dumps(params, indent=2, ensure_ascii=False, default=str))

        mean_metrics = cv_eval_one_paramset(
            X,
            y,
            folds,
            params,
            thresh=THRESH
        )

        score = mean_metrics["AUPRC_macro"]

        print(
            f"\n[TRIAL {trial.number:02d}] "
            f"score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []
    for t in study.trials:
        if t.value is None:
            continue

        row = {
            "trial": t.number,
            "AUPRC_macro": t.value,
            **t.params
        }

        row.update({
            k: v for k, v in t.user_attrs.items()
            if k.endswith("_macro")
        })

        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values(
        "AUPRC_macro",
        ascending=False
    )

    out_csv = "./optuna_singlemodel_RF_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

    # 额外保存标签列和特征列，便于后续核查
    label_out = "./used_label_columns_24.csv"
    feature_out = "./used_feature_columns.csv"

    pd.DataFrame({"label_col": label_cols}).to_csv(
        label_out,
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame({"feature_col": feat_cols}).to_csv(
        feature_out,
        index=False,
        encoding="utf-8-sig"
    )

    print("[SAVED]", label_out)
    print("[SAVED]", feature_out)


if __name__ == "__main__":
    main()

[INFO] Using feature file: ./Malodors_Rule&FG_features.xlsx


[I 2026-05-13 14:56:53,674] A new study created in memory with name: no-name-64d8427a-0107-4987-a44f-19b0c1a92c0f


[INFO] Column split mode: fixed-position
[INFO] SMILES column: Canonical_SMILES
[INFO] Label columns: columns 2-25, n=24
[INFO] Feature columns: columns 26-end, n=379

[INFO] Label columns:
  y01: alcoholic
  y02: aldehydic
  y03: almond
  y04: aromatic
  y05: burnt
  y06: cabbage
  y07: cheesy
  y08: cherry
  y09: chocolate
  y10: ethereal
  y11: fishy
  y12: fruity
  y13: garlic
  y14: gassy
  y15: green
  y16: ketonic
  y17: musty
  y18: pungent
  y19: sharp
  y20: solvent
  y21: sour
  y22: sulfurous
  y23: sweaty
  y24: sweet

[INFO] X shape=(3756, 379) (features=379) | y shape=(3756, 24) (labels=24)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] params:
{
  "n_jobs": -1,
  "random_state": 42,
  "n_estimators": 1441,
  "max_depth": 15,
  "max_features": 1.0,
  "min_samples_split": 13,
  "min_samples_leaf": 8,
  "bootstrap": false,
  "criterion": "gini"
}
    [FOLD 1] AUPRC=0.327244 | AUROC=0.793249 | Acc=0.938553 | P=0.565799 | R=0.211762 | Spec=0.965122
    [FOLD 2] AUPRC=0.320679 | AUROC=0.796581 | Acc=0.937528 | P=0.505864 | R=0.232063 | Spec=0.967505
    [FOLD 3] AUPRC=0.322194 | AUROC=0.775303 | Acc=0.940357 | P=0.573005 | R=0.224544 | Spec=0.969846
    [FOLD 4] AUPRC=0.329276 | AUROC=0.796882 | Acc=0.935863 | P=0.596025 | R=0.211827 | Spec=0.962968
    [FOLD 5] AUPRC=0.314688 | AUROC=0.794515 | Acc=0.936917 | P=0.451332 | R=0.218582 | Spec=0.963973

[TRIAL 00] score(AUPRC_macro)=0.322816 | AUROC=0.791306 | Acc=0.937844 | P=0.538405 | R=0.219756 | Spec=0.965883
[I 2026-05-13 14:58:26,525] Trial 0 finished with value: 0.3228164108441714 and parameters: {'max_depth': 15, 'n_estimators': 1441, 'max_features': 1.0,